In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.retail_training.pipeline_validation_results (
    pipeline_name STRING,
    validation_check STRING,
    validation_status STRING,
    checked_at TIMESTAMP
)
USING DELTA;


In [0]:

import uuid
from datetime import datetime, timezone

from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, LongType
)

# Task audit configuration
PIPELINE_NAME = "retail_data_pipeline"
TASK_NAME = "07_pipeline_validation"
AUDIT_TABLE = "workspace.retail_training.pipeline_audit_log"

RUN_ID = str(uuid.uuid4())
START_TIME = datetime.now(timezone.utc)
ROWS_PROCESSED = 0

print(f"Task audit ID: {RUN_ID}")


def write_audit_record(
    run_status,
    start_time,
    end_time,
    rows_processed=None,
    error_message=None
):
    audit_schema = StructType([
        StructField("pipeline_name", StringType(), False),
        StructField("task_name", StringType(), False),
        StructField("run_id", StringType(), True),
        StructField("run_status", StringType(), False),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("rows_processed", LongType(), True),
        StructField("error_message", StringType(), True),
        StructField("logged_at", TimestampType(), True)
    ])

    audit_record = [(
        PIPELINE_NAME,
        TASK_NAME,
        RUN_ID,
        run_status,
        start_time,
        end_time,
        rows_processed,
        error_message[:4000] if error_message else None,
        end_time
    )]

    audit_df = spark.createDataFrame(
        audit_record,
        schema=audit_schema
    )

    audit_df.write.format("delta").mode("append").saveAsTable(
        AUDIT_TABLE
    )


try:
    checked_at = datetime.now(timezone.utc)

    # Compare Silver metrics with Gold metrics
    validation_results = spark.sql("""
    WITH silver_metrics AS (
        SELECT
            COUNT(*) AS total_sales,
            COALESCE(SUM(quantity), 0) AS total_units,
            COALESCE(SUM(revenue), 0) AS total_revenue
        FROM workspace.retail_training.silver_sales
    ),
    gold_monthly_metrics AS (
        SELECT
            COALESCE(SUM(total_sales), 0) AS total_sales,
            COALESCE(SUM(total_quantity), 0) AS total_units,
            COALESCE(SUM(total_revenue), 0) AS total_revenue
        FROM workspace.retail_training.gold_monthly_sales
    ),
    gold_product_metrics AS (
        SELECT COALESCE(SUM(total_revenue), 0) AS total_revenue
        FROM workspace.retail_training.gold_product_sales
    ),
    gold_store_metrics AS (
        SELECT COALESCE(SUM(total_revenue), 0) AS total_revenue
        FROM workspace.retail_training.gold_store_sales
    )
    SELECT
        'Silver vs Gold Monthly' AS validation_check,
        CASE
            WHEN s.total_sales = m.total_sales
             AND s.total_units = m.total_units
             AND s.total_revenue = m.total_revenue
            THEN 'PASS' ELSE 'FAIL'
        END AS validation_status
    FROM silver_metrics s
    CROSS JOIN gold_monthly_metrics m

    UNION ALL

    SELECT
        'Silver vs Gold Product',
        CASE
            WHEN s.total_revenue = p.total_revenue
            THEN 'PASS' ELSE 'FAIL'
        END
    FROM silver_metrics s
    CROSS JOIN gold_product_metrics p

    UNION ALL

    SELECT
        'Silver vs Gold Store',
        CASE
            WHEN s.total_revenue = st.total_revenue
            THEN 'PASS' ELSE 'FAIL'
        END
    FROM silver_metrics s
    CROSS JOIN gold_store_metrics st
    """)

    # Add pipeline metadata and timestamp
    validation_rows = [
        (
            PIPELINE_NAME,
            row["validation_check"],
            row["validation_status"],
            checked_at
        )
        for row in validation_results.collect()
    ]

    ROWS_PROCESSED = len(validation_rows)

    validation_schema = StructType([
        StructField("pipeline_name", StringType(), False),
        StructField("validation_check", StringType(), False),
        StructField("validation_status", StringType(), False),
        StructField("checked_at", TimestampType(), False)
    ])

    validation_df = spark.createDataFrame(
        validation_rows,
        schema=validation_schema
    )

    # Save detailed results before handling failures
    validation_df.write.format("delta").mode("append").saveAsTable(
        "workspace.retail_training.pipeline_validation_results"
    )

    display(validation_df)

    failed_checks = sum(
        1 for row in validation_rows
        if row[2] == "FAIL"
    )

    print(f"Total validation checks: {len(validation_rows)}")
    print(f"Passed: {len(validation_rows) - failed_checks}")
    print(f"Failed: {failed_checks}")

    if failed_checks > 0:
        raise RuntimeError(
            f"Data validation failed: {failed_checks} check(s) failed. "
            "Review pipeline_validation_results."
        )

    print("SUCCESS: All data validation checks passed.")

except Exception as e:
    end_time = datetime.now(timezone.utc)

    try:
        write_audit_record(
            run_status="FAILED",
            start_time=START_TIME,
            end_time=end_time,
            rows_processed=ROWS_PROCESSED,
            error_message=str(e)
        )
        print("FAILED audit record written.")
    except Exception as audit_error:
        print(
            f"WARNING: Could not write failure audit record: "
            f"{audit_error}"
        )

    raise

else:
    end_time = datetime.now(timezone.utc)

    write_audit_record(
        run_status="SUCCESS",
        start_time=START_TIME,
        end_time=end_time,
        rows_processed=ROWS_PROCESSED,
        error_message=None
    )

    print(
        f"SUCCESS audit record written: "
        f"{ROWS_PROCESSED} validation checks completed."
    )


Task audit ID: 5329dda1-c020-497a-925c-b5855b5d13a1


pipeline_name,validation_check,validation_status,checked_at
retail_data_pipeline,Silver vs Gold Monthly,PASS,2026-10-21T03:02:24.994Z
retail_data_pipeline,Silver vs Gold Product,PASS,2026-10-21T03:02:24.994Z
retail_data_pipeline,Silver vs Gold Store,PASS,2026-10-21T03:02:24.994Z


Total validation checks: 3
Passed: 3
Failed: 0
SUCCESS: All data validation checks passed.
SUCCESS audit record written: 3 validation checks completed.
